# Optunaによるパラメータのオートチューニング

In [ ]:
!pip install -qq optuna kaggle-environments

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 14.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 56.2 MB

In [2]:
"""Colabで実行するパラメータ探索コード。"""

import importlib
import statistics

import optuna
from kaggle_environments import make

import main as strategy


# Colab上の最新のmain.pyを読み込む。
strategy = importlib.reload(strategy)

N_TRIALS = 100
MATCH_COUNT = 20
EVALUATION_SEEDS = list(range(MATCH_COUNT))


def objective(trial):
    """平均得点を返す。"""

    #購入を進める土地の目標区画数
    strategy.StrategyConfig.TARGET_LAND_COUNT = trial.suggest_int(
        "TARGET_LAND_COUNT",
        2,
        4,
    )

    #土地購入を許可する最低所持金
    strategy.StrategyConfig.LAND_PRICE = trial.suggest_int(
        "LAND_PRICE",
        3000,
        7000,
        step=500,
    )

    #購入を進める牛の目標頭数
    strategy.StrategyConfig.TARGET_COW_COUNT = trial.suggest_int(
        "TARGET_COW_COUNT",
        2,
        5,
    )

    #建設を進める牧草地の目標数
    strategy.StrategyConfig.TARGET_PASTURE_COUNT = trial.suggest_int(
        "TARGET_PASTURE_COUNT",
        2,
        5
    )

    rewards = []

    for episode_seed in EVALUATION_SEEDS:
        strategy.hire_controller = strategy.HireController()

        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": 720,
                "seed": episode_seed,
            },
            debug=True,
        )

        env.run([strategy.agent, strategy.agent])

        for state in env.steps[-1]:
            rewards.append(float(state.reward))

    return statistics.fmean(rewards)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

print("\n===== 最高平均得点 =====")
print(study.best_value)

print("\n===== main.pyへ手動設定する値 =====")

for name, value in study.best_params.items():
    print(f"{name} = {value}")

[I 2026-09-08 03:30:29,977] A new study created in memory with name: no-name-156c762d-b339-4f62-907f-359c47976bed


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-09-08 03:33:34,134] Trial 0 finished with value: 68282.225 and parameters: {'TARGET_LAND_COUNT': 3, 'LAND_PRICE': 7000, 'TARGET_COW_COUNT': 4, 'TARGET_PASTURE_COUNT': 4}. Best is trial 0 with value: 68282.225.
[I 2026-09-08 03:36:22,462] Trial 1 finished with value: 58193.025 and parameters: {'TARGET_LAND_COUNT': 2, 'LAND_PRICE': 3500, 'TARGET_COW_COUNT': 2, 'TARGET_PASTURE_COUNT': 5}. Best is trial 0 with value: 68282.225.
[I 2026-09-08 03:39:28,936] Trial 2 finished with value: 54181.025 and parameters: {'TARGET_LAND_COUNT': 3, 'LAND_PRICE': 6000, 'TARGET_COW_COUNT': 2, 'TARGET_PASTURE_COUNT': 5}. Best is trial 0 with value: 68282.225.
[I 2026-09-08 03:42:34,402] Trial 3 finished with value: 52875.275 and parameters: {'TARGET_LAND_COUNT': 4, 'LAND_PRICE': 3500, 'TARGET_COW_COUNT': 2, 'TARGET_PASTURE_COUNT': 2}. Best is trial 0 with value: 68282.225.
[I 2026-09-08 03:45:18,490] Trial 4 finished with value: 62911.95 and parameters: {'TARGET_LAND_COUNT': 2, 'LAND_PRICE': 5000, '